# SAR2Height Test Pipeline

This notebook provides a complete end-to-end pipeline for running SAR-to-height predictions using trained regression and diffusion models with ensemble prediction and uncertainty quantification.

## Pipeline Overview

The SAR2Height prediction pipeline follows these key steps:

1. **Setup**: Load dependencies and configure paths
2. **Data Loading**: Load preprocessed SAR2Height patches and statistics
3. **Model Loading**: Initialize trained regression and diffusion models
4. **Prediction Pipeline**: Run regression → diffusion ensemble → uncertainty analysis
5. **Metrics & Evaluation**: Calculate comprehensive evaluation metrics
6. **Visualization**: Generate publication-quality plots and analysis
7. **Results Export**: Save all results and visualizations

## Key Features

- **Ensemble Prediction**: Generate multiple diffusion samples for uncertainty quantification
- **Comprehensive Metrics**: Standard regression metrics, height-specific accuracy measures, spatial analysis
- **Advanced Visualization**: Input channels, prediction comparisons, ensemble analysis, statistical plots
- **Experiment Management**: Automated workflow execution with result organization and saving
- **Scientific Analysis**: Perfect for model comparison and uncertainty analysis in research papers

## Setup and Dependencies

### Install required packages if needed

In [1]:
import subprocess
import sys

def install_package(package):
    """Install a package using pip if not already installed."""
    try:
        __import__(package.split('==')[0])
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
        print(f"✓ {package} installed successfully")

# Install required packages for prediction pipeline
required_packages = [
    'seaborn>=0.13.0',
    'scikit-learn>=1.3.0',
    'scipy>=1.10.0',
    'h5netcdf>=1.6.0'
]

print("🔧 Installing required packages for prediction pipeline...")
for package in required_packages:
    try:
        install_package(package)
    except Exception as e:
        print(f"⚠️  Warning: Could not install {package}: {e}")

print("\n✅ Package installation complete!")

🔧 Installing required packages for prediction pipeline...
📦 Installing seaborn>=0.13.0...



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


✓ seaborn>=0.13.0 installed successfully
📦 Installing scikit-learn>=1.3.0...



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


✓ scikit-learn>=1.3.0 installed successfully
📦 Installing scipy>=1.10.0...



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


✓ scipy>=1.10.0 installed successfully
📦 Installing h5netcdf>=1.6.0...
✓ h5netcdf>=1.6.0 installed successfully

✅ Package installation complete!



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


### Standard library imports

In [2]:
import os
import sys
from pathlib import Path
import logging
import warnings
from typing import Dict, List, Optional, Tuple, Any
import time
from datetime import datetime

# Scientific computing imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr

# PyTorch imports
import torch
import torch.nn as nn

import json

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("✓ Standard imports loaded successfully")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🚀 Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

✓ Standard imports loaded successfully
🔥 PyTorch version: 2.7.1+cu126
🚀 Device available: CUDA


### Prediction Pipeline imports

In [3]:
# Add the prediction pipeline directory to Python path
pipeline_dir = Path('./5-sar2height/prediction_pipeline').resolve()
if str(pipeline_dir) not in sys.path:
    sys.path.insert(0, str(pipeline_dir))

# Import prediction pipeline modules
import_success = {}
import_errors = {}

prediction_modules = [
    ('data_manager', ['SAR2HeightDataManager']),
    ('regression_pipeline', ['SAR2HeightRegressionPipeline']), 
    ('diffusion_pipeline', ['SAR2HeightDiffusionPipeline']),
    ('ensemble_analyzer', ['EnsembleAnalyzer']),
    ('metrics_calculator', ['SAR2HeightMetrics']),
    ('visualizer', ['SAR2HeightVisualizer']),
    ('experiment_manager', ['ExperimentManager'])
]

for module_name, imports in prediction_modules:
    try:
        module = __import__(module_name)
        for import_name in imports:
            globals()[import_name] = getattr(module, import_name)
        import_success[module_name] = True
        print(f"✅ {module_name}: {', '.join(imports)}")
    except ImportError as e:
        import_errors[module_name] = str(e)
        import_success[module_name] = False
        print(f"❌ Failed to import {module_name}: {e}")

# Report import status
successful_imports = sum(import_success.values())
total_modules = len(import_success)

print(f"\n📦 Prediction Pipeline Import Status: {successful_imports}/{total_modules} modules loaded")

if successful_imports == total_modules:
    print("🎉 All prediction pipeline components loaded successfully!")
    print("🚀 Ready for SAR2Height prediction workflow!")
else:
    failed_modules = [name for name, success in import_success.items() if not success]
    print(f"⚠️  Failed modules: {', '.join(failed_modules)}")
    print("\n🔍 Import Error Details:")
    for module, error in import_errors.items():
        print(f"  • {module}: {error}")
    print("\n💡 Consider checking module dependencies and file locations")

✅ data_manager: SAR2HeightDataManager
✅ regression_pipeline: SAR2HeightRegressionPipeline
✅ diffusion_pipeline: SAR2HeightDiffusionPipeline
✅ ensemble_analyzer: EnsembleAnalyzer
✅ metrics_calculator: SAR2HeightMetrics
✅ visualizer: SAR2HeightVisualizer
✅ experiment_manager: ExperimentManager

📦 Prediction Pipeline Import Status: 7/7 modules loaded
🎉 All prediction pipeline components loaded successfully!
🚀 Ready for SAR2Height prediction workflow!


In [4]:
! ls /app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_regression/

UNet.0.44000.mdlus  UNet.0.51008.mdlus	   checkpoint.0.48000.pt
UNet.0.45008.mdlus  UNet.0.52000.mdlus	   checkpoint.0.49008.pt
UNet.0.46000.mdlus  UNet.0.53008.mdlus	   checkpoint.0.50000.pt
UNet.0.47008.mdlus  checkpoint.0.44000.pt  checkpoint.0.51008.pt
UNet.0.48000.mdlus  checkpoint.0.45008.pt  checkpoint.0.52000.pt
UNet.0.49008.mdlus  checkpoint.0.46000.pt  checkpoint.0.53008.pt
UNet.0.50000.mdlus  checkpoint.0.47008.pt


In [5]:
! ls /app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_diffusion/

EDMPrecondSuperResolution.0.10000.mdlus  checkpoint.0.10000.pt
EDMPrecondSuperResolution.0.1008.mdlus	 checkpoint.0.1008.pt
EDMPrecondSuperResolution.0.15008.mdlus  checkpoint.0.15008.pt
EDMPrecondSuperResolution.0.20000.mdlus  checkpoint.0.20000.pt
EDMPrecondSuperResolution.0.25008.mdlus  checkpoint.0.25008.pt
EDMPrecondSuperResolution.0.30000.mdlus  checkpoint.0.30000.pt
EDMPrecondSuperResolution.0.35008.mdlus  checkpoint.0.35008.pt
EDMPrecondSuperResolution.0.40000.mdlus  checkpoint.0.40000.pt
EDMPrecondSuperResolution.0.45008.mdlus  checkpoint.0.45008.pt
EDMPrecondSuperResolution.0.5008.mdlus	 checkpoint.0.5008.pt


## Configuration and Parameters

In [12]:
# =============================================================================
# CONFIGURATION PARAMETERS - SAR2Height Prediction Pipeline
# =============================================================================

# =============================================================================
# PATHS CONFIGURATION
# =============================================================================

# Base directories
BASE_DATA_DIR = Path("/app/data/sar2height/test/")
BASE_OUTPUT_DIR = Path("/app/outputs/sar2height/test_pipeline/")

# Data paths
DATA_FILE_PATH = BASE_DATA_DIR / "processed" / "preprocessed_patches" / "preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches.nc"
STATS_DIR = str(BASE_DATA_DIR / "processed" / "preprocessed_patches")

# Model checkpoint paths
REGRESSION_CHECKPOINT = "/app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_regression/UNet.0.53008.mdlus"
DIFFUSION_CHECKPOINT = "/app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_diffusion/EDMPrecondSuperResolution.0.45008.mdlus"

# =============================================================================
# OUTPUT DIRECTORY CONFIGURATION
# =============================================================================

# Extract dataset name from data file for organized directory structure
DATASET_NAME = DATA_FILE_PATH.stem  # e.g., "preprocessed_patches_full_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches"
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Hierarchical output structure:
# OUTPUT_DIR/
# └── {DATASET_NAME}/
#     └── {TIMESTAMP}/
#         ├── 1/  (sample index folders)
#         ├── 2/
#         ├── 3/
#         └── ...
DATASET_OUTPUT_DIR = BASE_OUTPUT_DIR / DATASET_NAME
TIMESTAMP_OUTPUT_DIR = DATASET_OUTPUT_DIR / TIMESTAMP
OUTPUT_DIR = str(TIMESTAMP_OUTPUT_DIR)

# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

# Experiment identification
EXPERIMENT_BASE_NAME = "sar2height_prediction"
EXPERIMENT_TIMESTAMP = TIMESTAMP

# Data sample selection - UPDATED TO SUPPORT MULTIPLE INDICES
SAMPLE_INDICES = list(range(0, 100)) #[0, 1, 2, 5, 10] # list(tange(0, 765)) # List of sample indices to analyze
# Alternative configurations:
# SAMPLE_INDICES = list(range(0, 20))  # First 20 samples
# SAMPLE_INDICES = [0, 5, 10, 15, 20, 25]  # Specific samples
# SAMPLE_INDICES = [42]  # Single sample (legacy compatibility)

# Input variables to use (must match training configuration)
SELECTED_VARIABLES = [
    'intensity_db',
    'intensity_percentile_rescaled'
]

# =============================================================================
# PIXEL RESOLUTION AND VISUALIZATION CONFIGURATION
# =============================================================================

# Actual pixel resolutions (for documentation and future use)
PIXEL_RESOLUTION_H = 1.0  # meters per pixel in height direction (rows)
PIXEL_RESOLUTION_W = 0.25  # meters per pixel in width direction (columns)

# Visualization aspect ratio correction
# Since H=1m and W=0.25m, the width direction is 4x higher resolution
# For proper aspect ratio in plots, we need to adjust the display
VISUALIZATION_ASPECT_RATIO_FACTOR = PIXEL_RESOLUTION_H / PIXEL_RESOLUTION_W  # 4.0
APPLY_ASPECT_RATIO_CORRECTION = True  # Whether to apply aspect ratio correction in plots

# Visualization display options
EQUAL_ASPECT_PLOTS = False  # Use equal aspect ratio (square pixels) - ignores real resolution
PHYSICAL_ASPECT_PLOTS = True  # Use physical aspect ratio (correct real-world proportions)

# Alternative aspect ratio settings (choose one approach)
ASPECT_RATIO_METHOD = "physical"  # Options: "physical", "equal", "auto"
# "physical": Use real pixel resolutions for correct geographic display
# "equal": Force square pixels (ignore resolution difference) 
# "auto": Let matplotlib decide

print(f"\n📐 Pixel Resolution Configuration:")
print(f"  Height direction: {PIXEL_RESOLUTION_H} m/pixel")
print(f"  Width direction: {PIXEL_RESOLUTION_W} m/pixel")
print(f"  Aspect ratio factor: {VISUALIZATION_ASPECT_RATIO_FACTOR:.1f}")
print(f"  Visualization method: {ASPECT_RATIO_METHOD}")
print(f"  Apply correction: {'ON' if APPLY_ASPECT_RATIO_CORRECTION else 'OFF'}")

# =============================================================================
# BATCH PROCESSING CONFIGURATION
# =============================================================================

# Model loading optimization
LOAD_MODELS_ONCE = True  # Load models once and reuse for all samples
CLEAR_CACHE_BETWEEN_SAMPLES = True  # Clear GPU cache between samples to prevent memory issues

# Parallel processing (future enhancement)
PARALLEL_PROCESSING = False  # Process multiple samples in parallel (not implemented yet)
MAX_PARALLEL_WORKERS = 4  # Maximum number of parallel workers

# Progress tracking
SHOW_PROGRESS_BAR = True  # Display progress bar for batch processing
SAVE_INTERMEDIATE_RESULTS = True  # Save results after each sample completion

# =============================================================================
# MODEL CONFIGURATION
# =============================================================================

# Model architecture configuration (must match training)
REGRESSION_CONFIG = {
    "N_grid_channels": 4,
    "gridtype": "sinusoidal", 
    "embedding_type": "zero",
    "model_channels": 128,
    "channel_mult": [1, 2, 2, 2, 2],
    "attn_resolutions": [28],
    "model_type": "SongUNetPosEmbd",
}

DIFFUSION_CONFIG = {
    # Model architecture from diffusion_normal.yaml (EXACT MATCH)
    "gridtype": "sinusoidal",
    "N_grid_channels": 4,
    "embedding_type": "zero",  # Changed from "positional" to "zero" like debug
    "model_channels": 128,
    "channel_mult": [1, 2, 2, 2, 2],
    "attn_resolutions": [28],
    "model_type": "SongUNetPosEmbd",
    
    # Model execution config (EXACT MATCH with training)
    "use_fp16": False,
    "checkpoint_level": 0,
    "hr_mean_conditioning": True,  # This is the ONLY parameter passed to ResidualLoss
}

# =============================================================================
# ENSEMBLE CONFIGURATION
# =============================================================================

# Ensemble prediction settings
ENSEMBLE_SIZE = 10  # Number of ensemble members to generate
ENSEMBLE_BASE_SEED = 42  # Base random seed for ensemble generation
DIFFUSION_STEPS = None  # Use default from model (or specify e.g., 50)

# Ensemble seed strategy for multiple samples
USE_DIFFERENT_SEEDS_PER_SAMPLE = False  # Use different base seeds for each sample
SEED_OFFSET_MULTIPLIER = 1000  # Offset between sample seeds (sample_idx * SEED_OFFSET_MULTIPLIER)

# =============================================================================
# ANALYSIS CONFIGURATION
# =============================================================================

# Visualization and output settings
SAVE_PLOTS = False  # Whether to save visualization plots
SAVE_DETAILED_RESULTS = False  # Whether to save detailed numerical results

# Figure quality settings
FIGURE_DPI = 150  # DPI for saved figures
FIGURE_FORMAT = 'png'  # Format for saved figures

# Cross-sample analysis
GENERATE_SUMMARY_REPORT = False  # Generate summary report across all samples
COMPARE_SAMPLES_VISUALIZATION = False  # Create comparative visualizations
SAVE_AGGREGATED_METRICS = False  # Save aggregated metrics across samples

# =============================================================================
# VISUALIZATION ASPECT RATIO CONFIGURATION
# =============================================================================

# Advanced aspect ratio control for visualizations
VISUALIZATION_CONFIG = {
    "aspect_ratio_factor": VISUALIZATION_ASPECT_RATIO_FACTOR,
    "apply_correction": APPLY_ASPECT_RATIO_CORRECTION,
    "method": ASPECT_RATIO_METHOD,
    "pixel_resolution_h": PIXEL_RESOLUTION_H,
    "pixel_resolution_w": PIXEL_RESOLUTION_W,
    "force_equal_aspect": EQUAL_ASPECT_PLOTS,
    "use_physical_aspect": PHYSICAL_ASPECT_PLOTS
}

# =============================================================================
# DIRECTORY STRUCTURE CONFIGURATION
# =============================================================================

# Directory naming and organization
SAMPLE_FOLDER_PREFIX = f"{EXPERIMENT_BASE_NAME}_sample_"  # Template Prefix for sample folders (e.g., "sample_")
RESULTS_FOLDER_NAME = "predictions"  # Name of predictions subfolder
ANALYSIS_FOLDER_NAME = "analysis"  # Name of analysis subfolder
VISUALIZATIONS_FOLDER_NAME = "visualizations"  # Name of visualizations subfolder
SUMMARY_FOLDER_NAME = "summary"  # Name of cross-sample summary folder

# File naming conventions
EXPERIMENT_NAME_TEMPLATE = f"{EXPERIMENT_BASE_NAME}_sample_{{sample_idx}}"  # Template for individual experiments
SUMMARY_NAME = f"{EXPERIMENT_BASE_NAME}_summary_{EXPERIMENT_TIMESTAMP}"  # Name for summary report

# =============================================================================
# DEVICE CONFIGURATION
# =============================================================================

# Compute device selection
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f"🚀 Using GPU: {torch.cuda.get_device_name()}")
else:
    DEVICE = 'cpu'
    print("💻 Using CPU")

# Memory management
CLEAR_GPU_CACHE = True  # Clear GPU cache between samples
GPU_MEMORY_FRACTION = 0.9  # Fraction of GPU memory to use

# =============================================================================
# DIRECTORY CREATION
# =============================================================================

# Create necessary output directories
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIMESTAMP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create sample-specific directories
SAMPLE_DIRECTORIES = {}
for sample_idx in SAMPLE_INDICES:
    sample_dir = TIMESTAMP_OUTPUT_DIR / f"{SAMPLE_FOLDER_PREFIX}{sample_idx}"
    sample_dir.mkdir(exist_ok=True)
    SAMPLE_DIRECTORIES[sample_idx] = sample_dir
    
    # Create subdirectories for each sample
    (sample_dir / RESULTS_FOLDER_NAME).mkdir(exist_ok=True)
    (sample_dir / ANALYSIS_FOLDER_NAME).mkdir(exist_ok=True)
    (sample_dir / VISUALIZATIONS_FOLDER_NAME).mkdir(exist_ok=True)

# Create summary directory for cross-sample analysis
SUMMARY_DIR = TIMESTAMP_OUTPUT_DIR / SUMMARY_FOLDER_NAME
SUMMARY_DIR.mkdir(exist_ok=True)

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

print(f"\n🔧 SAR2Height Multi-Sample Prediction Configuration Summary:")
print(f"  📁 Data file: {DATA_FILE_PATH.name}")
print(f"  📁 Dataset name: {DATASET_NAME}")
print(f"  📁 Stats directory: {STATS_DIR}")
print(f"  🤖 Regression checkpoint: {Path(REGRESSION_CHECKPOINT).name}")
print(f"  🌀 Diffusion checkpoint: {Path(DIFFUSION_CHECKPOINT).name}")
print(f"  📊 Output structure:")
print(f"    └── {BASE_OUTPUT_DIR.name}/")
print(f"        └── {DATASET_NAME}/")
print(f"            └── {TIMESTAMP}/")
for idx in SAMPLE_INDICES:
    print(f"                ├── {SAMPLE_FOLDER_PREFIX}{idx}/")
print(f"                └── {SUMMARY_FOLDER_NAME}/")
print(f"  🎯 Experiment timestamp: {EXPERIMENT_TIMESTAMP}")
print(f"  🔢 Sample indices: {SAMPLE_INDICES} ({len(SAMPLE_INDICES)} samples)")
print(f"  📈 Input variables: {SELECTED_VARIABLES}")
print(f"  🎲 Ensemble size: {ENSEMBLE_SIZE}")
print(f"  🔄 Load models once: {'ON' if LOAD_MODELS_ONCE else 'OFF'}")
print(f"  🖥️  Device: {DEVICE.upper()}")
print(f"  💾 Save plots: {'ON' if SAVE_PLOTS else 'OFF'}")
print(f"  📋 Generate summary: {'ON' if GENERATE_SUMMARY_REPORT else 'OFF'}")

# Validate critical paths
path_checks = [
    ('Data file', DATA_FILE_PATH),
    ('Stats directory', Path(STATS_DIR)),
    ('Regression checkpoint', Path(REGRESSION_CHECKPOINT)),
    ('Diffusion checkpoint', Path(DIFFUSION_CHECKPOINT))
]

print(f"\n🔍 Path Validation:")
all_paths_valid = True
for name, path in path_checks:
    if path.exists():
        print(f"  ✅ {name}: Found")
    else:
        print(f"  ❌ {name}: NOT FOUND - {path}")
        all_paths_valid = False

print(f"\n📁 Created Directories:")
print(f"  ✅ Base output: {BASE_OUTPUT_DIR}")
print(f"  ✅ Dataset folder: {DATASET_OUTPUT_DIR}")
print(f"  ✅ Timestamp folder: {TIMESTAMP_OUTPUT_DIR}")
print(f"  ✅ Sample directories: {len(SAMPLE_DIRECTORIES)} created")
print(f"  ✅ Summary directory: {SUMMARY_DIR}")

if all_paths_valid:
    print(f"\n🎉 All paths validated successfully! Ready to run multi-sample prediction pipeline.")
    print(f"📊 Will process {len(SAMPLE_INDICES)} samples with organized output structure.")
    print(f"📐 Aspect ratio correction: {VISUALIZATION_ASPECT_RATIO_FACTOR:.1f}x factor will be applied to visualizations.")
else:
    print(f"\n⚠️  Some paths are missing. Please update the configuration above.")

# =============================================================================
# UTILITY FUNCTIONS FOR DIRECTORY MANAGEMENT
# =============================================================================

def get_sample_output_dir(sample_idx: int) -> Path:
    """Get the output directory for a specific sample."""
    return SAMPLE_DIRECTORIES.get(sample_idx, TIMESTAMP_OUTPUT_DIR / f"{SAMPLE_FOLDER_PREFIX}{sample_idx}")

def get_sample_experiment_name(sample_idx: int) -> str:
    """Get the experiment name for a specific sample."""
    return EXPERIMENT_NAME_TEMPLATE.format(sample_idx=sample_idx)

def get_sample_ensemble_seed(sample_idx: int) -> int:
    """Get the ensemble base seed for a specific sample."""
    if USE_DIFFERENT_SEEDS_PER_SAMPLE:
        return ENSEMBLE_BASE_SEED + (sample_idx * SEED_OFFSET_MULTIPLIER)
    return ENSEMBLE_BASE_SEED

def get_sample_subdirectory(sample_idx: int, subdir_type: str) -> Path:
    """Get a specific subdirectory for a sample."""
    sample_dir = get_sample_output_dir(sample_idx)
    subdir_map = {
        'predictions': RESULTS_FOLDER_NAME,
        'analysis': ANALYSIS_FOLDER_NAME,
        'visualizations': VISUALIZATIONS_FOLDER_NAME
    }
    return sample_dir / subdir_map.get(subdir_type, subdir_type)

def get_visualization_aspect_ratio() -> float:
    """Get the aspect ratio factor for visualization correction."""
    return VISUALIZATION_ASPECT_RATIO_FACTOR if APPLY_ASPECT_RATIO_CORRECTION else 1.0

def get_visualization_config() -> Dict[str, Any]:
    """Get the complete visualization configuration."""
    return VISUALIZATION_CONFIG.copy()

print(f"\n🛠️  Utility functions ready for multi-sample processing.")


📐 Pixel Resolution Configuration:
  Height direction: 1.0 m/pixel
  Width direction: 0.25 m/pixel
  Aspect ratio factor: 4.0
  Visualization method: physical
  Apply correction: ON
🚀 Using GPU: Tesla V100-PCIE-32GB

🔧 SAR2Height Multi-Sample Prediction Configuration Summary:
  📁 Data file: preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches.nc
  📁 Dataset name: preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches
  📁 Stats directory: /app/data/sar2height/test/processed/preprocessed_patches
  🤖 Regression checkpoint: UNet.0.53008.mdlus
  🌀 Diffusion checkpoint: EDMPrecondSuperResolution.0.45008.mdlus
  📊 Output structure:
    └── test_pipeline/
        └── preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches/
            └── 20260115_140508/
                ├── sar2height_prediction_sample_0/
                ├── sar2height_prediction_sample_1/
                ├── sar2height_prediction_sample_2/
                ├── sar2height_pre

In [13]:
ds = xr.open_dataset(DATA_FILE_PATH)
ds_input = xr.open_dataset(DATA_FILE_PATH, group="input")
ds_output = xr.open_dataset(DATA_FILE_PATH, group="output")
ds_patch_metadata = xr.open_dataset(DATA_FILE_PATH, group="patch_metadata")

ds_patch_metadata

<xarray.Dataset> Size: 957kB
Dimensions:                        (sample: 1014)
Dimensions without coordinates: sample
Data variables: (12/19)
    patch_id                       (sample) int32 4kB ...
    grid_row_start                 (sample) int32 4kB ...
    grid_col_start                 (sample) int32 4kB ...
    bounds_left                    (sample) float64 8kB ...
    bounds_bottom                  (sample) float64 8kB ...
    bounds_right                   (sample) float64 8kB ...
    ...                             ...
    dsm_min                        (sample) float32 4kB ...
    dsm_max                        (sample) float32 4kB ...
    dsm_range                      (sample) float32 4kB ...
    feature_valid_percentage_min   (sample) float32 4kB ...
    feature_valid_percentage_mean  (sample) float32 4kB ...
    min_data_completeness          (sample) float32 4kB ...
Attributes:
    description:  Complete patch metadata for filtering
    n_patches:    1014

In [14]:
ls /app/outputs/sar2height/test_pipeline/

preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches/


In [15]:
! chmod -R 777 /app/outputs/sar2height/test_pipeline/

## Prediction Pipeline Execution

### Option 1: Complete Automated Workflow

In [16]:
# =============================================================================
# COMPLETE AUTOMATED MULTI-SAMPLE PREDICTION WORKFLOW
# =============================================================================

if successful_imports == total_modules and all_paths_valid:
    print("🚀 Starting Complete SAR2Height Multi-Sample Prediction Workflow")
    print("=" * 80)
    print(f"📊 Processing {len(SAMPLE_INDICES)} samples: {SAMPLE_INDICES}")
    print(f"📁 Dataset: {DATASET_NAME}")
    print(f"⏰ Timestamp: {EXPERIMENT_TIMESTAMP}")
    print(f"📐 Aspect ratio correction: {VISUALIZATION_ASPECT_RATIO_FACTOR:.1f}x")
    print("=" * 80)
    
    # Initialize containers for batch processing
    successful_experiments = {}
    failed_experiments = []
    batch_summary = {
        'total_samples': len(SAMPLE_INDICES),
        'successful_samples': 0,
        'failed_samples': 0,
        'processing_times': {},
        'model_performance': {},
        'visualization_counts': {},
        'errors': {}
    }
    
    # Model loading optimization - load once if enabled
    loaded_models = None
    if LOAD_MODELS_ONCE:
        print("🔧 Loading models once for batch processing...")
        try:
            # Create temporary experiment manager for model loading
            temp_experiment = ExperimentManager(
                experiment_name="temp_model_loader",
                output_dir=str(TIMESTAMP_OUTPUT_DIR),
                config={'visualization_config': VISUALIZATION_CONFIG}
            )
            
            # Setup data (using first sample for model configuration)
            first_sample = SAMPLE_INDICES[0]
            temp_experiment.setup_data(
                str(DATA_FILE_PATH), STATS_DIR, first_sample, SELECTED_VARIABLES
            )
            
            # Setup models
            models_loaded = temp_experiment.setup_models(
                REGRESSION_CONFIG, DIFFUSION_CONFIG, 
                REGRESSION_CHECKPOINT, DIFFUSION_CHECKPOINT
            )
            
            if models_loaded:
                loaded_models = {
                    'regression_pipeline': temp_experiment.regression_pipeline,
                    'diffusion_pipeline': temp_experiment.diffusion_pipeline,
                    'data_manager_template': temp_experiment.data_manager
                }
                print("✅ Models loaded successfully for batch processing")
            else:
                print("❌ Failed to load models - will load per sample")
                LOAD_MODELS_ONCE = False
                
        except Exception as e:
            print(f"❌ Model loading failed: {e}")
            print("🔄 Falling back to per-sample model loading")
            LOAD_MODELS_ONCE = False
            loaded_models = None
    
    # Process each sample
    for i, sample_idx in enumerate(SAMPLE_INDICES):
        print(f"\n{'='*60}")
        print(f"🔍 Processing Sample {i+1}/{len(SAMPLE_INDICES)}: Index {sample_idx}")
        print(f"{'='*60}")
        
        sample_start_time = time.time()
        
        try:
            # Get sample-specific paths and configuration
            sample_output_dir = get_sample_output_dir(sample_idx)
            sample_experiment_name = get_sample_experiment_name(sample_idx)
            sample_ensemble_seed = get_sample_ensemble_seed(sample_idx)
            
            print(f"📁 Sample output: {sample_output_dir}")
            print(f"🎯 Experiment name: {sample_experiment_name}")
            print(f"🎲 Ensemble seed: {sample_ensemble_seed}")
            
            # Create experiment manager for this sample
            experiment = ExperimentManager(
                experiment_name=sample_experiment_name,
                output_dir=str(sample_output_dir.parent),
                config={
                    'sample_idx': sample_idx,
                    'visualization_config': VISUALIZATION_CONFIG,
                    'aspect_ratio_factor': VISUALIZATION_ASPECT_RATIO_FACTOR,
                    'apply_aspect_correction': APPLY_ASPECT_RATIO_CORRECTION
                }
            )
            
            # Reuse loaded models if available
            if LOAD_MODELS_ONCE and loaded_models:
                print("🔄 Reusing pre-loaded models...")
                experiment.regression_pipeline = loaded_models['regression_pipeline']
                experiment.diffusion_pipeline = loaded_models['diffusion_pipeline']
                
                # Setup data for this specific sample
                data_setup_success = experiment.setup_data(
                    str(DATA_FILE_PATH), STATS_DIR, sample_idx, SELECTED_VARIABLES
                )
                
                if not data_setup_success:
                    raise Exception(f"Data setup failed for sample {sample_idx}")
                
                # Update model references in results
                experiment.results['models'] = {
                    'regression': experiment.regression_pipeline.get_model_info(),
                    'diffusion': experiment.diffusion_pipeline.get_model_info(),
                    'n_input_channels': len(SELECTED_VARIABLES),
                    'regression_checkpoint': REGRESSION_CHECKPOINT,
                    'diffusion_checkpoint': DIFFUSION_CHECKPOINT,
                    'reused_models': True
                }
                
                models_ready = True
            else:
                models_ready = False
            
            # Run complete experiment for this sample
            if models_ready:
                # Run remaining steps (skip model setup since already done)
                steps_success = (
                    experiment.run_predictions(
                        ENSEMBLE_SIZE, sample_ensemble_seed, DIFFUSION_STEPS
                    ) and
                    experiment.run_ensemble_analysis() and
                    experiment.calculate_metrics() and
                    # experiment.generate_visualizations(SAVE_PLOTS) and
                    experiment.save_results()
                )
            else:
                # Run complete workflow including model setup
                steps_success = experiment.run_complete_experiment(
                    data_file_path=str(DATA_FILE_PATH),
                    stats_dir=STATS_DIR,
                    regression_config=REGRESSION_CONFIG,
                    diffusion_config=DIFFUSION_CONFIG,
                    regression_checkpoint=REGRESSION_CHECKPOINT,
                    diffusion_checkpoint=DIFFUSION_CHECKPOINT,
                    sample_idx=sample_idx,
                    selected_variables=SELECTED_VARIABLES,
                    ensemble_size=ENSEMBLE_SIZE,
                    ensemble_base_seed=sample_ensemble_seed,
                    diffusion_steps=DIFFUSION_STEPS,
                    save_plots=SAVE_PLOTS
                )
            
            # Record processing time
            sample_processing_time = time.time() - sample_start_time
            batch_summary['processing_times'][sample_idx] = sample_processing_time
            
            if steps_success:
                print(f"✅ Sample {sample_idx} completed successfully in {sample_processing_time:.2f}s")
                
                # Store successful experiment
                successful_experiments[sample_idx] = experiment
                batch_summary['successful_samples'] += 1
                
                # Extract performance metrics
                if 'metrics' in experiment.results:
                    metrics = experiment.results['metrics'].get('model_performance', {})
                    batch_summary['model_performance'][sample_idx] = metrics
                
                # Count visualizations
                if 'visualizations' in experiment.results:
                    viz_info = experiment.results['visualizations']
                    viz_count = len(viz_info.get('figures_generated', []))
                    batch_summary['visualization_counts'][sample_idx] = viz_count
                
            else:
                raise Exception("Experiment workflow failed")
                
        except Exception as e:
            sample_processing_time = time.time() - sample_start_time
            print(f"❌ Sample {sample_idx} failed after {sample_processing_time:.2f}s: {e}")
            
            failed_experiments.append(sample_idx)
            batch_summary['failed_samples'] += 1
            batch_summary['errors'][sample_idx] = str(e)
            batch_summary['processing_times'][sample_idx] = sample_processing_time
        
        # Clear GPU cache between samples if enabled
        if CLEAR_CACHE_BETWEEN_SAMPLES and torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("🧹 GPU cache cleared")
        
        # Save intermediate progress
        if SAVE_INTERMEDIATE_RESULTS and (successful_experiments or failed_experiments):
            progress_file = SUMMARY_DIR / f"batch_progress_{EXPERIMENT_TIMESTAMP}.json"
            progress_data = {
                'processed_samples': i + 1,
                'total_samples': len(SAMPLE_INDICES),
                'successful_samples': list(successful_experiments.keys()),
                'failed_samples': failed_experiments,
                'current_sample': sample_idx,
                'timestamp': datetime.now().isoformat(),
                'processing_times': batch_summary['processing_times']
            }
            
            with open(progress_file, 'w') as f:
                json.dump(progress_data, f, indent=2, default=str)
    
    # Calculate total batch processing time
    total_batch_time = sum(batch_summary['processing_times'].values())
    batch_summary['total_processing_time'] = total_batch_time
    batch_summary['average_time_per_sample'] = total_batch_time / len(SAMPLE_INDICES) if SAMPLE_INDICES else 0
    
    # Generate comprehensive batch summary
    print("\n" + "=" * 80)
    print("📊 BATCH PROCESSING SUMMARY")
    print("=" * 80)
    
    print(f"📈 Overall Results:")
    print(f"  • Total samples processed: {len(SAMPLE_INDICES)}")
    print(f"  • Successful: {batch_summary['successful_samples']}")
    print(f"  • Failed: {batch_summary['failed_samples']}")
    print(f"  • Success rate: {(batch_summary['successful_samples']/len(SAMPLE_INDICES)*100):.1f}%")
    print(f"  • Total processing time: {total_batch_time:.2f} seconds")
    print(f"  • Average time per sample: {batch_summary['average_time_per_sample']:.2f} seconds")
    
    if successful_experiments:
        print(f"\n✅ Successful Samples: {list(successful_experiments.keys())}")
        
        # Performance summary across successful samples
        if batch_summary['model_performance']:
            print(f"\n📈 Model Performance Summary (RMSE):")
            
            reg_rmses = [perf.get('regression_rmse') for perf in batch_summary['model_performance'].values() if perf.get('regression_rmse')]
            diff_rmses = [perf.get('diffusion_rmse') for perf in batch_summary['model_performance'].values() if perf.get('diffusion_rmse')]
            
            if reg_rmses:
                print(f"  🤖 Regression - Mean: {np.mean(reg_rmses):.3f}m, Std: {np.std(reg_rmses):.3f}m")
            if diff_rmses:
                print(f"  🌀 Diffusion - Mean: {np.mean(diff_rmses):.3f}m, Std: {np.std(diff_rmses):.3f}m")
        
        # Visualization summary
        total_visualizations = sum(batch_summary['visualization_counts'].values())
        print(f"\n🎨 Visualization Summary:")
        print(f"  • Total visualizations generated: {total_visualizations}")
        print(f"  • Average per sample: {total_visualizations/len(successful_experiments):.1f}")
    
    if failed_experiments:
        print(f"\n❌ Failed Samples: {failed_experiments}")
        print(f"   Errors:")
        for sample_idx, error in batch_summary['errors'].items():
            print(f"   • Sample {sample_idx}: {error}")
    
    print(f"\n📁 Results Structure:")
    print(f"  📊 Dataset folder: {DATASET_OUTPUT_DIR}")
    print(f"  ⏰ Timestamp folder: {TIMESTAMP_OUTPUT_DIR}")
    for sample_idx in successful_experiments.keys():
        sample_dir = get_sample_output_dir(sample_idx)
        print(f"  📁 Sample {sample_idx}: {sample_dir}")
    if successful_experiments:
        print(f"  📋 Summary: {SUMMARY_DIR}")
    
    # Save complete batch summary
    if SAVE_DETAILED_RESULTS:
        summary_file = SUMMARY_DIR / f"complete_batch_summary_{EXPERIMENT_TIMESTAMP}.json"
        
        # Prepare serializable summary
        serializable_summary = batch_summary.copy()
        # Remove non-serializable experiment objects
        serializable_summary['successful_sample_indices'] = list(successful_experiments.keys())
        serializable_summary['failed_sample_indices'] = failed_experiments
        serializable_summary['configuration'] = {
            'dataset_name': DATASET_NAME,
            'sample_indices': SAMPLE_INDICES,
            'ensemble_size': ENSEMBLE_SIZE,
            'visualization_config': VISUALIZATION_CONFIG,
            'load_models_once': LOAD_MODELS_ONCE,
            'selected_variables': SELECTED_VARIABLES
        }
        
        with open(summary_file, 'w') as f:
            json.dump(serializable_summary, f, indent=2, default=str)
        
        print(f"\n💾 Complete batch summary saved: {summary_file.name}")
    
    # Final status
    if batch_summary['successful_samples'] == len(SAMPLE_INDICES):
        print("\n🎉 ALL SAMPLES PROCESSED SUCCESSFULLY!")
        print("✨ Multi-sample prediction pipeline completed without errors!")
        completed_experiments = successful_experiments
    elif batch_summary['successful_samples'] > 0:
        print(f"\n⚠️  PARTIAL SUCCESS: {batch_summary['successful_samples']}/{len(SAMPLE_INDICES)} samples completed")
        print("🔧 Check failed samples and retry if needed")
        completed_experiments = successful_experiments
    else:
        print("\n❌ ALL SAMPLES FAILED")
        print("🚨 Check configuration and error messages above")
        completed_experiments = None
    
    print("=" * 80)

else:
    print("❌ Cannot run automated multi-sample workflow:")
    if successful_imports < total_modules:
        print(f"   • Missing imports: {total_modules - successful_imports} modules failed")
    if not all_paths_valid:
        print(f"   • Invalid paths: Check configuration above")
    print("\n💡 Fix the issues above or use manual processing.")
    completed_experiments = None

🚀 Starting Complete SAR2Height Multi-Sample Prediction Workflow
📊 Processing 100 samples: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
📁 Dataset: preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches
⏰ Timestamp: 20260115_140508
📐 Aspect ratio correction: 4.0x
🔧 Loading models once for batch processing...
🎨 Visualizer initialized with aspect ratio correction: False
🚀 SAR2Height Experiment Manager initialized: temp_model_loader
📁 Results will be saved to: /app/outputs/sar2height/test_pipeline/preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches/20260115_140508/temp_model_loader
📊 Predictions: /app/outputs/sar2h

INFO - Loaded model state dictionary /app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_regression/UNet.0.53008.mdlus to device cuda
INFO - Loaded checkpoint file /app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_regression/checkpoint.0.53008.pt to device cuda


✓ SAR2Height Regression model loaded successfully from epoch 53008
✓ Model set to evaluation mode
✓ Model moved to device: cuda
🌀 Setting up diffusion pipeline...
🎯 EXACT COPY: Initializing SAR2HeightDiffusionPipeline like train.py
✓ Deterministic inference configured (seed: 42)
🔧 Creating SAR2Height diffusion model EXACTLY like train.py...
✓ SAR2Height Model created: total_in_channels=7
   Dataset channels: 2
   HR mean conditioning: True
   Grid channels: 4
   Output channels: 1


INFO - Loaded model state dictionary /app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_diffusion/EDMPrecondSuperResolution.0.45008.mdlus to device cuda
INFO - Loaded checkpoint file /app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_diffusion/checkpoint.0.45008.pt to device cuda


✓ SAR2Height Diffusion model loaded from epoch 45008
🔧 Creating diffusion loss function with regression network...
🔧 Creating ResidualLoss for SAR2Height EXACTLY like train.py...
   Loading SAR2Height regression network from: /app/checkpoints/sar2height/intensity_db-intensity_percentile_rescaled/checkpoints_regression/UNet.0.53008.mdlus
✓ SAR2Height regression network loaded successfully
✓ SAR2Height ResidualLoss created with:
   hr_mean_conditioning: True
   P_mean: 0.0 (class default)
   P_std: 1.2 (class default)
   sigma_data: 0.5 (class default)
✓ Diffusion loss function created successfully
✓ Model pipelines setup completed successfully
✅ Models loaded successfully for batch processing

🔍 Processing Sample 1/100: Index 0
📁 Sample output: /app/outputs/sar2height/test_pipeline/preprocessed_patches_ICEYE_X8_SLC_SLH_54750_20210503T175109_1014patches/20260115_140508/sar2height_prediction_sample_0
🎯 Experiment name: sar2height_prediction_sample_0
🎲 Ensemble seed: 42
🎨 Visualizer initia